In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/processed/temiz_veri.csv")
df.shape   # (7043, 21) çıkmalı

(7043, 21)

In [2]:
# Müşteri süresi grupları (include_lowest: tenure=0 olanlar da gruba girsin)
df['tenure_group'] = pd.cut(df['tenure'], bins=[0,12,24,48,72],
    labels=['0-1 yıl','1-2 yıl','2-4 yıl','4-6 yıl'], include_lowest=True)

# Aylık ücret grupları
df['charge_group'] = pd.cut(df['MonthlyCharges'], bins=[0,35,70,120],
    labels=['Düşük','Orta','Yüksek'])

# Aylık ortalama harcama (tenure=0 ise 0)
df['avg_monthly'] = np.where(df['tenure'] > 0, df['TotalCharges'] / df['tenure'], 0)

# Toplam hizmet sayısı
services = ['OnlineSecurity','OnlineBackup','DeviceProtection',
            'TechSupport','StreamingTV','StreamingMovies']
df['service_count'] = (df[services] == 'Yes').sum(axis=1)

df[['tenure_group','charge_group']].isna().sum()   # ikisi de 0 olmalı

tenure_group    0
charge_group    0
dtype: int64

## Feature engineering: neden ürettim?

- **tenure_group:** Müşteri süresi çok geniş bir aralıkta, gruplayınca yeni/eski müşteri farkı görülür.
- **charge_group:** Ücretleri düşük/orta/yüksek diye karşılaştırmak, research question'daki "ücretlendirme" için kolay.
- **avg_monthly:** Müşterinin gerçekte aylık ne ödediğini gösterir.
- **service_count:** Altı hizmeti tek sayıya özetler, müşterinin firmaya bağlılığını ölçer.

In [3]:
keep = ['tenure','tenure_group','MonthlyCharges','charge_group','TotalCharges',
        'avg_monthly','service_count','InternetService','TechSupport',
        'Contract','PaperlessBilling','PaymentMethod','Churn']

df_sel = df[keep].copy()
df_sel.shape

(7043, 13)

## Feature selection: Kaldı / Elendi / Neden

| Sütun | Durum | Neden |
|---|---|---|
| customerID | Elendi | Her müşteriye özel bir kod, analize bilgi katmıyor |
| gender | Elendi | Research question ile ilgisiz |
| SeniorCitizen | Elendi | Research question ile ilgisiz |
| Partner | Elendi | Research question ile ilgisiz |
| Dependents | Elendi | Research question ile ilgisiz |
| PhoneService | Elendi | Soru internet ve teknik desteğe odaklı |
| MultipleLines | Elendi | Soru internet ve teknik desteğe odaklı |
| OnlineSecurity | Elendi | service_count içinde özetlendi |
| OnlineBackup | Elendi | service_count içinde özetlendi |
| DeviceProtection | Elendi | service_count içinde özetlendi |
| StreamingTV | Elendi | service_count içinde özetlendi |
| StreamingMovies | Elendi | service_count içinde özetlendi |
| tenure | Kaldı | Müşteri süresi, churn ile ilişkili olabilir |
| tenure_group | Kaldı | Müşteri süresinin gruplanmış hali |
| MonthlyCharges | Kaldı | Ücretlendirme (research question) |
| charge_group | Kaldı | Ücretlendirme, gruplanmış hali |
| TotalCharges | Kaldı | Ücretlendirme (research question) |
| avg_monthly | Kaldı | Gerçek aylık harcama |
| service_count | Kaldı | Müşterinin aldığı hizmet sayısı |
| InternetService | Kaldı | İnternet hizmeti türü (research question) |
| TechSupport | Kaldı | Teknik destek (research question) |
| Contract | Kaldı | Sözleşme türü, ücretlendirmeyle bağlantılı |
| PaperlessBilling | Kaldı | Fatura şekli, ücretlendirmeyle bağlantılı |
| PaymentMethod | Kaldı | Ödeme şekli, ücretlendirmeyle bağlantılı |
| Churn | Kaldı | Hedef değişken |

In [5]:
from sklearn.preprocessing import LabelEncoder

df_enc = df_sel.copy()

# 1) Label encoding: Yes/No ve sıralı gruplar
for col in ['PaperlessBilling','tenure_group','charge_group']:
    df_enc[col] = LabelEncoder().fit_transform(df_enc[col].astype(str))

# 2) One-hot encoding: sırası olmayan kategoriler
df_enc = pd.get_dummies(df_enc, columns=['InternetService','TechSupport','Contract'], dtype=int)

# 3) Frequency encoding: PaymentMethod
freq = df_enc['PaymentMethod'].value_counts(normalize=True)
df_enc['PaymentMethod_freq'] = df_enc['PaymentMethod'].map(freq)
df_enc = df_enc.drop(columns=['PaymentMethod'])

df_enc.to_csv("../../data/processed/kodlanmis_veri.csv", index=False)
df_enc.head()

,tenure,tenure_group,MonthlyCharges,charge_group,TotalCharges,avg_monthly,service_count,PaperlessBilling,Churn,InternetService_DSL,InternetService_Fiber optic,InternetService_No,TechSupport_No,TechSupport_No internet service,TechSupport_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_freq
0,1,0,29.85,0,29.85,29.850000,1,1,0,1,0,0,1,0,0,1,0,0,0.335794
1,34,2,56.95,1,1889.50,55.573529,2,0,0,1,0,0,1,0,0,0,1,0,0.228880
2,2,0,53.85,1,108.15,54.075000,2,1,1,1,0,0,1,0,0,1,0,0,0.228880
3,45,2,42.30,1,1840.75,40.905556,3,0,0,1,0,0,0,0,1,0,1,0,0.219225
4,2,0,70.70,2,151.65,75.825000,0,1,1,0,1,0,1,0,0,1,0,0,0.335794


In [6]:
df_enc.dtypes

tenure                               int64
tenure_group                         int64
MonthlyCharges                     float64
charge_group                         int64
TotalCharges                       float64
avg_monthly                        float64
service_count                        int64
PaperlessBilling                     int64
Churn                                int64
InternetService_DSL                  int64
InternetService_Fiber optic          int64
InternetService_No                   int64
TechSupport_No                       int64
TechSupport_No internet service      int64
TechSupport_Yes                      int64
Contract_Month-to-month              int64
Contract_One year                    int64
Contract_Two year                    int64
PaymentMethod_freq                 float64
dtype: object

In [7]:
# Not: Veri normal dağılmıyor (bkz. aşağıdaki 'Yöntem seçimi' notu). Z-Score bu veride ideal değil, karşılaştırma için uygulandı.
from sklearn.preprocessing import MinMaxScaler, StandardScaler

cols = ['tenure','MonthlyCharges','TotalCharges','avg_monthly','service_count']

df_scaled = df_enc.copy()
df_scaled[[c+'_norm' for c in cols]] = MinMaxScaler().fit_transform(df_enc[cols])
df_scaled[[c+'_std' for c in cols]] = StandardScaler().fit_transform(df_enc[cols])

df_scaled.to_csv("../../data/processed/olceklenmis_veri.csv", index=False)
df_scaled.head()

,tenure,tenure_group,MonthlyCharges,charge_group,TotalCharges,avg_monthly,service_count,PaperlessBilling,Churn,InternetService_DSL,...,tenure_norm,MonthlyCharges_norm,TotalCharges_norm,avg_monthly_norm,service_count_norm,tenure_std,MonthlyCharges_std,TotalCharges_std,avg_monthly_std,service_count_std
0,1,0,29.85,0,29.85,29.850000,1,1,0,1,...,0.013889,0.115423,0.003437,0.245881,0.166667,-1.277445,-1.160323,-0.992611,-1.151302,-0.561776
1,34,2,56.95,1,1889.50,55.573529,2,0,0,1,...,0.472222,0.385075,0.217564,0.457772,0.333333,0.066327,-0.259629,-0.172165,-0.301458,-0.020519
2,2,0,53.85,1,108.15,54.075000,2,1,1,1,...,0.027778,0.354229,0.012453,0.445428,0.333333,-1.236724,-0.362660,-0.958066,-0.350966,-0.020519
3,45,2,42.30,1,1840.75,40.905556,3,0,0,1,...,0.625000,0.239303,0.211951,0.336949,0.500000,0.514251,-0.746535,-0.193672,-0.786053,0.520738
4,2,0,70.70,2,151.65,75.825000,0,1,1,0,...,0.027778,0.521891,0.017462,0.624588,0.000000,-1.236724,0.197365,-0.938874,0.367602,-1.103033


## Normalization ve standardization
- **Normalization (MinMax):** Değerleri 0 ile 1 arasına sıkıştırır.
- **Standardization:** Ortalamayı 0, standart sapmayı 1 yapar.
- Sütunların ölçekleri çok farklı (tenure 0-72, TotalCharges binlerle), bu yüzden ikisi de uygulandı.
- Orijinal sütunlar korundu, yeni sütunlar `_norm` ve `_std` ile eklendi.

## Yöntem seçimi: neden bu normalizasyon?
- İstatistik notebook'unda gördüğümüz gibi **hiçbir sayısal sütun normal dağılmıyor**:
  - TotalCharges sağa çarpık (skewness 0.96)
  - tenure ve MonthlyCharges basık (kurtosis −1.39 ve −1.26)
- **Z-Score (StandardScaler)** ortalama ve standart sapmaya dayanır, veri normal dağılıma uygunsa doğru sonuç verir. Çarpık TotalCharges'ta ortalama kuyruk tarafına kaydığı için **bu veride doğru sonuç vermez.**
- **Min-Max** normal ya da düzgün (uniform) dağılımlarda iyi çalışır; değerleri 0–1 arasına taşıdığı için ölçekleri karşılaştırmada kullanılabilir, ama çarpıklığı düzeltmez.
- Min-Max ve Z-Score'u **karşılaştırma amacıyla** uyguladık. Çarpık veri için en uygun yöntem **Robust Scaling**: ortalama yerine medyan ve çeyreklikler (IQR) üzerinden hesap yaptığı için çarpıklıktan ve uç değerlerden etkilenmez.
- **Log transformation** veriyi sıkıştırsa da gerçek farkları erittiği ve dağılımı bozduğu için tercih etmedik.
